# 01. Dunder methods

**Tier:** Intermediate. **Why you'd use this:** A bag of items should work with `len`, `in`, and indexing like a built-in collection. Dunder methods are the hooks those operations call.

**After this lesson you can:** implement `__len__`, `__getitem__`, and `__contains__`; explain the `__eq__`/`__hash__` contract; use `__len__` for truthiness.
About 25 minutes. **Uses:** [Classes](../11-classes/01-classes-lesson.ipynb)


## Quick review

1. Why catch a custom exception by type?
2. What does `super()` follow?

<details><summary>Answers</summary>

Callers can handle one failure mode. The MRO.
</details>


## The idea

Names like `__len__` and `__eq__` are dunder methods. The language calls them for built-ins and operators. Defining `__eq__` disables hashing unless you also define a compatible `__hash__`. `__bool__` or `__len__` decides truthiness.

<details><summary>Explain more</summary>

`len(obj)` becomes `obj.__len__()`. `item in obj` can use `__contains__`. Empty length makes an instance false in `if`.
</details>


## See it work

Give a count length and membership.


In [1]:
class Count:
    def __init__(self, n):
        self.n = n

    def __len__(self):
        # 1. Built-ins call this dunder.
        return self.n

    def __bool__(self):
        return self.n != 0


print(len(Count(3)))
print(bool(Count(0)))
print(bool(Count(2)))

3
False
True


## Check yourself

Answer these before you run the cell.

1. After defining only `__eq__`, can instances go in a set?
2. Does `len(Count(0))` make the instance false if there is no `__bool__`?


In [2]:
class Box:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return isinstance(other, Box) and self.value == other.value


print(Box(1) == Box(1))
print(hash(Box(1)))

True


TypeError: unhashable type: 'Box'

<details><summary>Answers</summary>

No. `__hash__` becomes `None`. Yes. Truthiness falls back to `__len__`.
</details>


## Why it works this way

Built-ins and syntax call dunder methods on your objects. `len(x)` looks for `x.__len__`. `item in x` looks for `__contains__`, or falls back to iteration. Indexing calls `__getitem__`. That protocol style is how user types plug into the language without special-case syntax. Deep dive: [Attribute lookup](../../internals/08-attributes/01-lookup-lesson.ipynb).

Equality and hashing are a contract. If you define `__eq__`, Python sets `__hash__` to `None` on the class so instances become unhashable by default. Equal objects must have equal hashes when both are hashable; otherwise a dict or set can lose them. Define `__hash__` only for immutable value objects, and derive it from the same fields `__eq__` uses.

Truthiness walks `__bool__` if present, else `__len__`, else defaults to true for instances. An empty container that defines `__len__` is false in an `if` without any extra work. That is why `if items:` works for your type when length is zero.

Cost of these dunders is one method call from the built-in. The design reason is one language for many types: `len`, `in`, and `for` stay stable while your storage changes. Evidence: `len(Count(3))` uses `__len__`, equal boxes with `__eq__` are unhashable until you opt in carefully, and `bool(Count(0))` follows length.

Protocols keep call sites stable while storage changes. A bag can switch from a list to a dict internally and still support `len` and `in` if the dunders keep their meaning. Prefer the smallest set of dunders that make the type feel native. Reach for `__iter__` when `for` should work. Reach for `__getitem__` when indexing matters. Document whether your equality is identity-like or value-like so callers know whether hashing is available.

When you implement `__hash__`, ask whether the object can change after it is placed in a set. If yes, leave it unhashable. If no, hash exactly the fields that `__eq__` compares. That discipline prevents the silent loss of set members after a field mutates. Truthiness should match the user's idea of empty. Length zero is the usual signal for containers.


In [3]:
class Count:
    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n


print(len(Count(3)))
print(bool(Count(0)))


class Box:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return isinstance(other, Box) and self.value == other.value


print(Box(1) == Box(1))
print(Box.__hash__ is None)


class Pocket:
    def __init__(self, items):
        self.items = items

    def __contains__(self, item):
        return item in self.items

    def __getitem__(self, index):
        return self.items[index]


print("a" in Pocket(["a", "b"]))
print(Pocket(["a", "b"])[1])

3
False
True
True
True
b


## When to use it

Use dunders when you want built-ins and syntax to work on your type. Keep `__eq__` and `__hash__` in sync for value objects. Use `__len__` or `__bool__` for truthiness. Don't put mutable equal objects in sets. Don't implement every dunder; implement the ones your API needs. If a type only needs iteration, implement `__iter__` before a pile of other hooks.


## Common mistake

Defining `__eq__` and then putting instances in a set without a hash, or writing a hash that ignores fields `__eq__` uses.


In [4]:
class Box:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return isinstance(other, Box) and self.value == other.value


print({Box(1), Box(1)})

TypeError: cannot use 'Box' as a set element (unhashable type: 'Box')

Leave mutable boxes unhashable, or freeze the value and define `__hash__` from the same fields.


In [5]:
class Box:
    def __init__(self, value):
        self.value = value

    def __eq__(self, other):
        return isinstance(other, Box) and self.value == other.value

    def __hash__(self):
        return hash(self.value)


print(len({Box(1), Box(1)}))
print(Box(1) == Box(1))

1
True


## Try it

Add `__iter__` to `Pocket` so `list(Pocket(items))` works. Which protocol does `for` use, and do you still need `__getitem__`?


## Recap

- Built-ins call dunder methods on your objects.
- Defining `__eq__` disables hashing until you opt in safely.
- `__bool__` or `__len__` decides truthiness.

**Next:** [practice](01-dunder-practice.ipynb)
